# Kumu Data Preparation — Jupyter Template

This notebook cleans an event-registration table and exports three Kumu-ready datasets:

1. **Org–Event (bipartite)** — Organizations connected to specific Events
2. **Org–Year (affiliation)** — Organizations connected to Years
3. **Org–Org (projection)** — Organizations connected to each other when they co-attend at least N events/years

**Outputs:** six CSV files (elements/connections for each view) written to an `out/` folder.

**Minimum columns in your input:**
- `Organization` (required)
- `Event` (e.g., "May 2005"). If you don't have `Year`, the notebook will try to extract it from `Event`.

**Optional columns:** `Year`, `SNA_Category`, `City`, `State`, `ZIP`, `FirstName`, `LastName`.

> Tip: If your data is in `.ods/.xlsx`, this notebook will *try* to read it. If that fails in your environment, export to `.csv` and update the path below.


In [5]:
# ---- 1) Setup: paths & parameters ----
from pathlib import Path

# Point to your input file here. If reading .ods/.xlsx fails, convert to CSV and update the path.
INPUT_PATH = r'C:\Users\charu\OneDrive\Desktop\Info Viz\Final Project\All_Conferences_Data_V5.ods'  # change to your file
OUTPUT_DIR = Path(r'C:\Users\charu\OneDrive\Desktop\Info Viz\Final Project\Client Data\Data')   # where CSV exports will be written
INCLUDE_PEOPLE = False              # set True to include person nodes & person→org edges (can get large)
ORGORG_MIN_CO = 2                   # threshold for Org–Org co-attendance edges

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
INPUT_PATH, OUTPUT_DIR

('C:\\Users\\charu\\OneDrive\\Desktop\\Info Viz\\Final Project\\All_Conferences_Data_V5.ods',
 WindowsPath('C:/Users/charu/OneDrive/Desktop/Info Viz/Final Project/Client Data/Data'))

In [7]:
# ---- 2) Imports ----
import re
import numpy as np
import pandas as pd

def _try_read_table(path: Path) -> pd.DataFrame:
    path = Path(path)
    suf = path.suffix.lower()
    if suf == '.csv':
        return pd.read_csv(path, dtype=str, keep_default_na=False)
    # try spreadsheet readers
    try:
        if suf in ('.xlsx', '.xls'):
            return pd.read_excel(path, dtype=str)
        if suf == '.ods':
            # requires 'odfpy' — may not exist in all environments
            return pd.read_excel(path, engine='odf', dtype=str)
    except Exception as e:
        print('\n⚠️ Unable to read spreadsheet directly:', e)
        print('→ Export your file to CSV and update INPUT_PATH above.')
        raise

df = _try_read_table(INPUT_PATH)
print('Loaded shape:', df.shape)
df.head(3)

Loaded shape: (6410, 18)


,UniqueID,Salutation,First,Last,SNA Category - ORIGINAL,SNA Category - CLEANED,Title,Title - CLEANED,Organization - ORIGINAL,Organization - CLEANED,Active / Inactive,Address,City,State,Zip,Conference,Month,Year
0,1,Ms.,Linda,Cooer,Other,Other,NaN,NaN,,NaN,NaN,1133 Ferdinand Ave,Forest Park,IL,60130,2000-05-01 00:00:00,5,2000
1,2,Ms.,Smita,Khatri,Other,Other,NaN,NaN,,NaN,NaN,4055 N. Saint Louis,Chicago,IL,60618,2000-05-01 00:00:00,5,2000
2,3,Ms.,Alpana,Patel,Other,Other,NaN,NaN,,NaN,NaN,5114 Arrowhead Dr.,Oak Forest,IL,60452,2000-05-01 00:00:00,5,2000


In [9]:
# --- Map your columns to the template's expected columns ---

import pandas as pd
import numpy as np

# 1) Organization
org_clean  = df.get('Organization - CLEANED')
org_orig   = df.get('Organization - ORIGINAL')
df['Organization'] = (
    org_clean.fillna('').astype(str).str.strip()
    .where(lambda s: s.ne(''), org_orig.fillna('').astype(str).str.strip())
)

# 2) Person (optional; harmless if you keep INCLUDE_PEOPLE=False)
df['FirstName'] = df.get('First', '').astype(str).str.strip()
df['LastName']  = df.get('Last', '').astype(str).str.strip()

# 3) Location
df['City']  = df.get('City',  '').astype(str).str.strip()
df['State'] = df.get('State', '').astype(str).str.strip()
# file shows 'Zip' not 'ZIP'
df['ZIP']   = df.get('Zip',   '').astype(str).str.strip()

# 4) Category
cat_clean = df.get('SNA Category - CLEANED')
cat_orig  = df.get('SNA Category - ORIGINAL')
df['SNA_Category'] = (
    cat_clean.fillna('').astype(str).str.strip()
    .where(lambda s: s.ne(''), cat_orig.fillna('').astype(str).str.strip())
)

# 5) Event label + Year
# Prefer the explicit datetime in 'Conference' if present; else use Month+Year.
month_name = {
    1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
    7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'
}

conf = pd.to_datetime(df.get('Conference', pd.NaT), errors='coerce')
yr_col = pd.to_numeric(df.get('Year', np.nan), errors='coerce')
mo_col = pd.to_numeric(df.get('Month', np.nan), errors='coerce')

# Year: prefer explicit Year; else from Conference
df['Year'] = (
    yr_col
    .where(~yr_col.isna(), conf.dt.year)
    .astype('Int64')
)

# Event label “Mon YYYY”
event_from_conf = conf.dt.strftime('%b %Y')
event_from_my   = mo_col.map(month_name).fillna('').astype(str).str.strip() + ' ' + df['Year'].astype('Int64').astype(str)
df['Event'] = event_from_conf.fillna('').where(event_from_conf.notna(), event_from_my).str.strip()

# 6) Final quick sanity
need = ['Organization','Event','Year','SNA_Category','City','State','ZIP','FirstName','LastName']
print('After mapping, empties (should be low for Organization/Event):')
print(df[need].isna().sum())
df[need].head()


After mapping, empties (should be low for Organization/Event):
Organization    0
Event           0
Year            0
SNA_Category    0
City            0
State           0
ZIP             0
FirstName       0
LastName        0
dtype: int64


,Organization,Event,Year,SNA_Category,City,State,ZIP,FirstName,LastName
0,,May 2000,2000,Other,Forest Park,IL,60130,Linda,Cooer
1,,May 2000,2000,Other,Chicago,IL,60618,Smita,Khatri
2,,May 2000,2000,Other,Oak Forest,IL,60452,Alpana,Patel
3,,May 2000,2000,Other,Chicago,IL,60643,Randolph,Thomas
4,19th District Youth Net,May 2000,2000,Other,Chicago,IL,60618,Lisa,Bodey


## 3) Cleaning helpers
Light normalization for `Organization`, extraction of `Year`, and whitespace cleanup.

In [11]:
def norm_text(x):
    if pd.isna(x) or str(x).strip() == '':
        return ''
    s = re.sub(r"\s+", " ", str(x)).strip()
    return s

def norm_org(x):
    s = norm_text(x)
    s = s.replace('&', 'and')
    s = re.sub(r"[.,]", "", s)
    s = re.sub(r"\s+", " ", s)
    return s

def coalesce_year(row):
    # Priority: explicit Year → else parse from Event
    if 'Year' in row and str(row['Year']).strip():
        m = re.findall(r"(19|20)\d{2}", str(row['Year']))
        if m:
            return int(m[0] if isinstance(m, str) else m[0])
    if 'Event' in row and str(row['Event']).strip():
        m = re.findall(r"(19|20)\d{2}", str(row['Event']))
        if m:
            # If regex returns tuples, pull the full match
            if isinstance(m[0], tuple):
                return int(''.join(m[0]))
            return int(m[0])
    return np.nan

def ensure_columns(df, required):
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

## 4) Normalize columns

In [17]:
required = ['Organization', 'Event']
ensure_columns(df, required)

for col in ['Year','SNA_Category','City','State','ZIP','FirstName','LastName']:
    if col not in df.columns:
        df[col] = ''

df['Organization_norm'] = df['Organization'].apply(norm_org)
df['Event_norm'] = df['Event'].apply(norm_text)
df['Year_norm'] = df.apply(coalesce_year, axis=1)

# If still missing but Year is a plain 4-digit
mask_plain_year = df['Year_norm'].isna() & df['Year'].astype(str).str.fullmatch(r'\d{4}')
df.loc[mask_plain_year, 'Year_norm'] = df.loc[mask_plain_year, 'Year'].astype(int)

print('After normalization:', df.shape)
df.sample(5, random_state=1)

After normalization: (6410, 27)


,UniqueID,Salutation,First,Last,SNA Category - ORIGINAL,SNA Category - CLEANED,Title,Title - CLEANED,Organization - ORIGINAL,Organization - CLEANED,...,Year,Organization,FirstName,LastName,ZIP,SNA_Category,Event,Organization_norm,Event_norm,Year_norm
2054,2055,Dr.,Edward,Gordon,Resource,Resource,President,President,Imperial Consulting Corporation,Imperial Consulting Corporation,...,2005,Imperial Consulting Corporation,Edward,Gordon,60611,Resource,May 2005,Imperial Consulting Corporation,May 2005,20
147,148,Ms.,Cheryl,Morace,Government,Government,School-to-Careers,School-to-Careers,Natchez-Adams County Economic Development Auth...,Natchez Adams County Economic Development Auth...,...,2000,Natchez Adams County Economic Development Auth...,Cheryl,Morace,39121,Government,May 2000,Natchez Adams County Economic Development Auth...,May 2000,20
5672,5698,Mr.,Diane,Kelly,Program,Program,NaN,NaN,Cook County Juvenile Probation Department,Cook County Juvenile Probation Department,...,1998,Cook County Juvenile Probation Department,Diane,Kelly,60612,Program,Nov 1998,Cook County Juvenile Probation Department,Nov 1998,19
4106,4114,NaN,Toni,"Ellison, MA",college,College,VISTA Training Coordinator,Coordinator,Illinois Campus Compact,Illinois Campus Compact,...,2014,Illinois Campus Compact,Toni,"Ellison, MA",60604,College,Nov 2014,Illinois Campus Compact,Nov 2014,20
4825,4851,Ms.,Carlos,Drazen,College,College,NaN,NaN,"Dept. of Disability and Human Development, UIC",Dept of Disability and Human Development UIC,...,1996,Dept of Disability and Human Development UIC,Carlos,Drazen,60608,College,May 1996,Dept of Disability and Human Development UIC,May 1996,19


In [21]:
# Recompute Year_norm with full 4-digit extraction
import re
import numpy as np
import pandas as pd

def extract_year_full(text: str):
    m = re.search(r"\b((?:19|20)\d{2})\b", str(text))
    return int(m.group(1)) if m else pd.NA

def _coalesce_after_norm(row):
    y2 = extract_year_full(row.get('Year', ''))
    if pd.notna(y2): return y2
    return extract_year_full(row.get('Event', ''))

df['Year_norm'] = df.apply(_coalesce_after_norm, axis=1).astype('Int64')

# If Year is exactly 4 digits, keep it
mask_plain_year = df['Year_norm'].isna() & df['Year'].astype(str).str.fullmatch(r"\d{4}", na=False)
df.loc[mask_plain_year, 'Year_norm'] = df.loc[mask_plain_year, 'Year'].astype(int)

df[['Event','Year','Year_norm']].head(10)


,Event,Year,Year_norm
0,May 2000,2000,2000
1,May 2000,2000,2000
2,May 2000,2000,2000
3,May 2000,2000,2000
4,May 2000,2000,2000
5,May 2000,2000,2000
6,May 2000,2000,2000
7,May 2000,2000,2000
8,May 2000,2000,2000
9,May 2000,2000,2000


In [23]:
df.head(5)

,UniqueID,Salutation,First,Last,SNA Category - ORIGINAL,SNA Category - CLEANED,Title,Title - CLEANED,Organization - ORIGINAL,Organization - CLEANED,...,Year,Organization,FirstName,LastName,ZIP,SNA_Category,Event,Organization_norm,Event_norm,Year_norm
0,1,Ms.,Linda,Cooer,Other,Other,NaN,NaN,,NaN,...,2000,,Linda,Cooer,60130,Other,May 2000,,May 2000,2000
1,2,Ms.,Smita,Khatri,Other,Other,NaN,NaN,,NaN,...,2000,,Smita,Khatri,60618,Other,May 2000,,May 2000,2000
2,3,Ms.,Alpana,Patel,Other,Other,NaN,NaN,,NaN,...,2000,,Alpana,Patel,60452,Other,May 2000,,May 2000,2000
3,4,Mr.,Randolph,Thomas,Other,Other,NaN,NaN,,NaN,...,2000,,Randolph,Thomas,60643,Other,May 2000,,May 2000,2000
4,5,Ms.,Lisa,Bodey,Other,Other,NaN,NaN,19th District Youth Net,19th District Youth Net,...,2000,19th District Youth Net,Lisa,Bodey,60618,Other,May 2000,19th District Youth Net,May 2000,2000


## 5) Builders — produce Elements/Connections for each view

In [25]:
def build_org_event(df, include_people=False):
    elements, connections = [], []
    # Organizations
    for org, g in df.groupby('Organization_norm', dropna=False):
        if not org:
            continue
        years = sorted(y for y in g['Year_norm'].dropna().tolist())
        elements.append({
            'id': f'ORG::{org}',
            'label': org,
            'type': 'Organization',
            'group': g['SNA_Category'].dropna().mode().iat[0] if not g['SNA_Category'].dropna().empty else '',
            'city': g['City'].dropna().mode().iat[0] if 'City' in g and not g['City'].dropna().empty else '',
            'state': g['State'].dropna().mode().iat[0] if 'State' in g and not g['State'].dropna().empty else '',
            'zip': g['ZIP'].dropna().mode().iat[0] if 'ZIP' in g and not g['ZIP'].dropna().empty else '',
            'participation_count': len(g.index),
            'years_attended': ','.join(map(str, years)) if years else '',
            'first_year': min(years) if years else '',
            'last_year': max(years) if years else ''
        })
    # Events
    for ev in sorted(set(e for e in df['Event_norm'].tolist() if e)):
        elements.append({
            'id': f'EV::{ev}',
            'label': ev,
            'type': 'Event',
            'group': 'Event',
            'city': '', 'state': '', 'zip': '',
            'participation_count': int(df[df['Event_norm'] == ev].shape[0]),
            'years_attended': '', 'first_year': '', 'last_year': ''
        })
    # Org—Event edges
    for _, r in df.iterrows():
        org, ev = r['Organization_norm'], r['Event_norm']
        if not org or not ev:
            continue
        connections.append({
            'from': f'ORG::{org}', 'to': f'EV::{ev}',
            'type': 'Attended', 'label': f"{r.get('Year_norm', '')}",
            'year': r.get('Year_norm', ''), 'weight': 1
        })
    # Optional people
    if include_people and 'FirstName' in df.columns and 'LastName' in df.columns:
        existing_ids = set(x['id'] for x in elements)
        for _, r in df.iterrows():
            first, last = norm_text(r.get('FirstName','')), norm_text(r.get('LastName',''))
            if not first and not last:
                continue
            pid = f'PERSON::{first} {last}'.strip()
            if pid not in existing_ids:
                elements.append({'id': pid, 'label': f'{first} {last}'.strip(), 'type': 'Person', 'group': 'Person',
                                 'city': norm_text(r.get('City','')), 'state': norm_text(r.get('State','')), 'zip': norm_text(r.get('ZIP','')),
                                 'participation_count': '', 'years_attended': '', 'first_year': '', 'last_year': ''})
                existing_ids.add(pid)
            org = r['Organization_norm']
            if org:
                connections.append({'from': pid, 'to': f'ORG::{org}', 'type': 'Affiliated-with', 'label': '',
                                   'year': r.get('Year_norm',''), 'weight': 1})
    import pandas as pd
    return pd.DataFrame(elements).drop_duplicates(subset=['id']), pd.DataFrame(connections)

def build_org_year(df):
    elements, connections = [], []
    # Orgs
    for org, g in df.groupby('Organization_norm', dropna=False):
        if not org:
            continue
        years = sorted(y for y in g['Year_norm'].dropna().tolist())
        elements.append({
            'id': f'ORG::{org}', 'label': org, 'type': 'Organization',
            'group': g['SNA_Category'].dropna().mode().iat[0] if not g['SNA_Category'].dropna().empty else '',
            'city': g['City'].dropna().mode().iat[0] if 'City' in g and not g['City'].dropna().empty else '',
            'state': g['State'].dropna().mode().iat[0] if 'State' in g and not g['State'].dropna().empty else '',
            'zip': g['ZIP'].dropna().mode().iat[0] if 'ZIP' in g and not g['ZIP'].dropna().empty else '',
            'participation_count': len(g.index),
            'years_attended': ','.join(map(str, years)) if years else '',
            'first_year': min(years) if years else '', 'last_year': max(years) if years else ''
        })
    # Years
    years = sorted(set(y for y in df['Year_norm'].tolist() if y == y))
    for y in years:
        elements.append({'id': f'YEAR::{int(y)}', 'label': str(int(y)), 'type': 'Year', 'group': 'Year',
                         'city':'','state':'','zip':'', 'participation_count': int(df[df['Year_norm']==y].shape[0]),
                         'years_attended':'','first_year':'','last_year':''})
    # Unique org–year pairs, sum weights
    pairs = df[['Organization_norm','Year_norm']].dropna()
    pairs = pairs[(pairs['Organization_norm']!='') & (pairs['Year_norm'].notna())]
    pairs['weight'] = 1
    pairs = pairs.groupby(['Organization_norm','Year_norm'], as_index=False)['weight'].sum()
    for _, r in pairs.iterrows():
        connections.append({'from': f"ORG::{r['Organization_norm']}", 'to': f"YEAR::{int(r['Year_norm'])}",
                            'type': 'Participated-in', 'label': str(int(r['Year_norm'])),
                            'year': int(r['Year_norm']), 'weight': int(r['weight'])})
    import pandas as pd
    return pd.DataFrame(elements).drop_duplicates(subset=['id']), pd.DataFrame(connections)

def build_orgorg(df, min_co=2):
    elements, connections = [], []
    # Elements: orgs
    for org, g in df.groupby('Organization_norm', dropna=False):
        if not org:
            continue
        years = sorted(y for y in g['Year_norm'].dropna().tolist())
        elements.append({'id': f'ORG::{org}', 'label': org, 'type':'Organization',
                         'group': g['SNA_Category'].dropna().mode().iat[0] if not g['SNA_Category'].dropna().empty else '',
                         'city': g['City'].dropna().mode().iat[0] if 'City' in g and not g['City'].dropna().empty else '',
                         'state': g['State'].dropna().mode().iat[0] if 'State' in g and not g['State'].dropna().empty else '',
                         'zip': g['ZIP'].dropna().mode().iat[0] if 'ZIP' in g and not g['ZIP'].dropna().empty else '',
                         'participation_count': len(g.index), 'years_attended': ','.join(map(str, years)) if years else '',
                         'first_year': min(years) if years else '', 'last_year': max(years) if years else ''})
    # Co-attendance by bucket (Event if present, else Year)
    dfp = df.copy()
    dfp['bucket'] = np.where(dfp['Event_norm'] != '', dfp['Event_norm'], dfp['Year_norm'].astype(str))
    dfp = dfp[dfp['Organization_norm'] != '']
    from itertools import combinations
    co_counts = {}
    for bucket, g in dfp.groupby('bucket'):
        orgs = sorted(set(g['Organization_norm'].tolist()))
        for a, b in combinations(orgs, 2):
            key = tuple(sorted((a, b)))
            co_counts[key] = co_counts.get(key, 0) + 1
    for (a, b), w in co_counts.items():
        if w >= min_co:
            connections.append({'from': f'ORG::{a}', 'to': f'ORG::{b}', 'type':'Co-attended',
                               'label': f'{w} shared events/years', 'year': '', 'weight': int(w)})
    import pandas as pd
    return pd.DataFrame(elements).drop_duplicates(subset=['id']), pd.DataFrame(connections)

## 6) Build each view & export CSVs

In [27]:
el1, co1 = build_org_event(df, include_people=INCLUDE_PEOPLE)
el2, co2 = build_org_year(df)
el3, co3 = build_orgorg(df, min_co=ORGORG_MIN_CO)

el1.to_csv(OUTPUT_DIR / 'elements_org_event.csv', index=False)
co1.to_csv(OUTPUT_DIR / 'connections_org_event.csv', index=False)
el2.to_csv(OUTPUT_DIR / 'elements_org_year.csv', index=False)
co2.to_csv(OUTPUT_DIR / 'connections_org_year.csv', index=False)
el3.to_csv(OUTPUT_DIR / 'elements_orgorg.csv', index=False)
co3.to_csv(OUTPUT_DIR / 'connections_orgorg.csv', index=False)

print('Wrote files to', OUTPUT_DIR.resolve())
el2.head(), co2.head()

Wrote files to C:\Users\charu\OneDrive\Desktop\Info Viz\Final Project\Client Data\Data


(                                      id                             label  \
 0          ORG::100 Black Men of Chicago          100 Black Men of Chicago   
 1                    ORG::10711 S Church                    10711 S Church   
 2                 ORG::111th Street YMCA                 111th Street YMCA   
 3                     ORG::16th Ward Org                     16th Ward Org   
 4  ORG::17th District Advisory Committee  17th District Advisory Committee   
 
            type    group     city state    zip  participation_count  \
 0  Organization  Program  Chicago    IL  60012                    2   
 1  Organization    Other  Chicago    IL  60643                    1   
 2  Organization  Program      nan   nan    nan                    1   
 3  Organization    Other  Chicago    IL  60643                    1   
 4  Organization    Other  Chicago    IL  60625                    1   
 
   years_attended first_year last_year  
 0      2003,2013       2003      2013  
 1      

## 7) Quick quality checks
- Do `Organization_norm` and `Event_norm` look clean?
- Any suspicious duplicates in organizations? Consider adding a manual alias table if needed.
- Are there years failing to parse? You can adjust the `coalesce_year` function above.

In [30]:
print('Unique orgs:', df['Organization_norm'].nunique())
print('Unique events:', df['Event_norm'].nunique())
print('Unique years:', df['Year_norm'].nunique())

df[['Organization','Organization_norm']].drop_duplicates().head(10)

Unique orgs: 1778
Unique events: 41
Unique years: 21


,Organization,Organization_norm
0,,
4,19th District Youth Net,19th District Youth Net
6,Abbott,Abbott
8,Alliance for Community Peace Near North Minist...,Alliance for Community Peace Near North Minist...
9,Alliance for Community Peace A Unique After Sc...,Alliance for Community Peace A Unique After Sc...
12,AmeriCorps,AmeriCorps
13,Ariel Education,Ariel Education
14,ARK of St Sabina 6th District Youth Net,ARK of St Sabina 6th District Youth Net
15,Asian Human Services SAFE Mentoring Program,Asian Human Services SAFE Mentoring Program
18,Associated Colleges of Illinois,Associated Colleges of Illinois


## 8) Kumu import tips (styling & legends)
- **Elements**: size by `participation_count`; color by `group` (SNA Category) or by `type` for bipartite clarity.
- **Connections**: label by `type` or `weight` (Org–Org); thickness by `weight` (Org–Org).
- **Legends**: add a Color legend (by `group` or `type`), a Size legend (by `participation_count`), and a Connection thickness legend (by `weight`).
- **Filters**: filter by `type` or `group`; add a `year` filter on connections to time-slice.
- **Layouts**: Org–Year/Event bipartite layout; Org–Org force-directed with a threshold of `weight ≥ 2` to reveal core clusters.

In [1]:
import pandas as pd
from pathlib import Path

out = Path(r'C:\Users\charu\OneDrive\Desktop\Info Viz\Final Project\Client Data\Data')  # change me

el = pd.read_csv(out/'elements_org_event.csv', dtype=str)
co = pd.read_csv(out/'connections_org_event.csv', dtype=str)

print('Types in elements_org_event:', el['type'].value_counts(dropna=False))
print('Connection types:', co['type'].value_counts(dropna=False))

# Assert there are only Org + Event in elements, and only Attended connections
assert set(el['type'].unique()) <= {'Organization','Event'}
assert set(co['type'].unique()) == {'Attended'}

# If you ever find stray types, you can hard-filter and re-save:
el_clean = el[el['type'].isin(['Organization','Event'])].copy()
co_clean = co.copy()  # all Attended already
el_clean.to_csv(out/'elements_org_event_PURE.csv', index=False)
co_clean.to_csv(out/'connections_org_event_PURE.csv', index=False)
print('Wrote _PURE files.')


Types in elements_org_event: type
Organization    1777
Event             41
Name: count, dtype: int64
Connection types: type
Attended    6100
Name: count, dtype: int64
Wrote _PURE files.


In [9]:
import pandas as pd, re
from pathlib import Path

# ========= 1) EDIT THESE TWO LINES =========
INPUT_PATH = r"C:\Users\charu\OneDrive\Desktop\Info Viz\Final Project\All_Conferences_Data_V5.ods"   # or .xlsx/.csv
OUTPUT_DIR = r"C:\Users\charu\OneDrive\Desktop\Info Viz\Final Project\Client Data\Data"                   # where CSVs will be written
# ==========================================

# ---- helpers ----
def try_read(path, sheet=0):
    p = str(path).lower()
    if p.endswith(".csv"):  return pd.read_csv(path, dtype=str)
    if p.endswith(".xlsx"): return pd.read_excel(path, sheet_name=sheet, dtype=str)
    if p.endswith(".ods"):  return pd.read_excel(path, sheet_name=sheet, engine="odf", dtype=str)
    raise ValueError("Unsupported file type (use .ods/.xlsx/.csv)")

def pick_col(df, candidates, required=True):
    """Pick first matching column name (case-insensitive) from candidates."""
    by_lower = {c.strip().lower(): c for c in df.columns}
    for cand in candidates:
        if cand.strip().lower() in by_lower:
            return by_lower[cand.strip().lower()]
    if required:
        raise KeyError(f"Missing any of {candidates}. Found: {list(df.columns)}")
    return None

def clean_txt(s): 
    return re.sub(r"\s+", " ", str(s)).strip()

def extract_year(text):
    m = re.search(r"\b((?:19|20)\d{2})\b", str(text))
    return int(m.group(1)) if m else pd.NA

# ---- load data ----
out = Path(OUTPUT_DIR); out.mkdir(parents=True, exist_ok=True)
raw = try_read(INPUT_PATH).copy()

# ---- pick the right columns from your sheet (robust to header variants) ----
org_col   = pick_col(raw, [
    "Organization - CLEANED", "Organization CLEANED",
    "Organization", "Organisation", "Organization - ORIGINAL"
])
event_col = pick_col(raw, [
    "Event", "Conference", "Event Name", "Conference Name"
])
sna_col   = pick_col(raw, [
    "SNA_Category - CLEANED", "SNA Category - CLEANED", "SNA_Category", "SNA Category"
], required=False)
year_col  = pick_col(raw, ["Year", "year"], required=False)

print("Columns selected:")
print("  Organization:", org_col)
print("  Event       :", event_col)
print("  SNA Category:", sna_col)
print("  Year        :", year_col)

# ---- minimal normalized table (NO PERSON rows) ----
df = pd.DataFrame({
    "Organization": raw[org_col].map(clean_txt),
    "Event"       : raw[event_col].map(clean_txt),
    "SNA_Category": raw[sna_col].map(clean_txt) if sna_col else "",
    "Year_raw"    : raw[year_col] if year_col else ""
})

# keep only attendance rows (org & event must exist) -> prevents any Person rows
df = df[(df["Organization"] != "") & (df["Event"] != "")].copy()

# robust year (optional; used for Org–Year only)
df["Year_norm"] = df.apply(
    lambda r: (extract_year(r["Year_raw"]) if pd.notna(r["Year_raw"]) and str(r["Year_raw"]).strip() != ""
               else extract_year(r["Event"])),
    axis=1
).astype("Int64")

# IDs
df["ORG_ID"]  = "ORG::"  + df["Organization"]
df["EV_ID"]   = "EV::"   + df["Event"]
df["YEAR_ID"] = "YEAR::" + df["Year_norm"].astype(str)

# participation & span (for sizing)
org_counts = (df.groupby("Organization").size()
                .rename("participation_count").reset_index())
org_span = (df.dropna(subset=["Year_norm"])
              .groupby("Organization")["Year_norm"]
              .agg(first_year="min", last_year="max").reset_index())
org_meta = (org_counts.merge(org_span, on="Organization", how="left")
                      .merge(df[["Organization","SNA_Category"]].drop_duplicates(),
                             on="Organization", how="left"))

# ========= A) BIPARTITE: Org–Event (NO year on edges) =========
els_org = (org_meta.assign(
    id=lambda x: "ORG::" + x["Organization"],
    label=lambda x: x["Organization"],
    type="Organization",
    group=lambda x: x["SNA_Category"]
)[["id","label","type","group","participation_count","first_year","last_year"]]
 .drop_duplicates())

els_evt = (df[["Event"]].drop_duplicates()
             .assign(id=lambda x: "EV::" + x["Event"],
                     label=lambda x: x["Event"],
                     type="Event", group="")
             [["id","label","type","group"]])

elements_org_event = pd.concat([els_org, els_evt], ignore_index=True).drop_duplicates()

# one edge per Org–Event pair, no extra date field -> Kumu can't print 2001/2002 on edges
connections_org_event = (df[["ORG_ID","EV_ID"]].drop_duplicates()
                           .rename(columns={"ORG_ID":"from","EV_ID":"to"})
                           .assign(type="Attended", weight=1))

elements_org_event.to_csv(out/"elements_org_event.csv", index=False)
connections_org_event.to_csv(out/"connections_org_event.csv", index=False)

print("\nOrg–Event: elements =", len(elements_org_event), " connections =", len(connections_org_event))

# ========= B) MULTI-COLORED OPTION 1: Org–Year =========
year_nodes = (df.dropna(subset=["Year_norm"])[["Year_norm"]].drop_duplicates()
                .assign(id=lambda x: "YEAR::" + x["Year_norm"].astype(str),
                        label=lambda x: x["Year_norm"].astype(str),
                        type="Year", group="")
                [["id","label","type","group"]])

elements_org_year = pd.concat([els_org, year_nodes], ignore_index=True).drop_duplicates()

connections_org_year = (df.dropna(subset=["Year_norm"])[["ORG_ID","YEAR_ID"]]
                          .drop_duplicates()
                          .rename(columns={"ORG_ID":"from","YEAR_ID":"to"})
                          .assign(type="Participated-in", weight=1))

elements_org_year.to_csv(out/"elements_org_year.csv", index=False)
connections_org_year.to_csv(out/"connections_org_year.csv", index=False)

print("Org–Year  : elements =", len(elements_org_year), " connections =", len(connections_org_year))

# ========= B alt) MULTI-COLORED OPTION 2: Org–Org (co-attendance) =========
pairs = (df[["Event","ORG_ID"]].drop_duplicates()
           .merge(df[["Event","ORG_ID"]].drop_duplicates(), on="Event")
           .query("ORG_ID_x < ORG_ID_y")
           .groupby(["ORG_ID_x","ORG_ID_y"]).size()
           .rename("weight").reset_index())

elements_orgorg = els_org.copy()
connections_orgorg = (pairs.rename(columns={"ORG_ID_x":"from","ORG_ID_y":"to"})
                           .assign(type="Co-attended"))

elements_orgorg.to_csv(out/"elements_orgorg.csv", index=False)
connections_orgorg.to_csv(out/"connections_orgorg.csv", index=False)

print("Org–Org   : elements =", len(elements_orgorg), " connections =", len(connections_orgorg))
print("\nWrote CSVs to:", out)


Columns selected:
  Organization: Organization - CLEANED
  Event       : Conference
  SNA Category: SNA Category - CLEANED
  Year        : Year

Org–Event: elements = 2059  connections = 3719
Org–Year  : elements = 2039  connections = 3187
Org–Org   : elements = 2018  connections = 153076

Wrote CSVs to: C:\Users\charu\OneDrive\Desktop\Info Viz\Final Project\Client Data\Data
